# HANDS-ON 01 | Primeros pasos con Apache Spark y PySpark
**Curso:** Base de Datos Avanzadas y Big Data · **Unidad III:** Big Data · **Semana:** 11  
**Docente:** Dr. Eric Gustavo Coronel Castillo · **Duración:** 4 horas  
**Integrantes del grupo:** 1. __________ 2. __________ 3. __________ 4. __________

> **Propósito:** poner en funcionamiento Spark, leer datos y diagnosticar su calidad antes de proponer un análisis. Las celdas señaladas **RETO** no contienen la solución.

**Datos:** `registros_operativos_simulados.csv` (28 filas ficticias creadas exclusivamente para aprendizaje; no son mediciones oficiales).

## 0. Preparación del entorno
Se sugiere **Google Colab**. Ejecuta la celda siguiente una sola vez por sesión y reinicia el entorno de ejecución **si el instalador lo indica**. Si trabajas en Jupyter local con PySpark instalado, omite la instalación.

Referencia: https://spark.apache.org/docs/latest/api/python/getting_started/install.html

In [ ]:
%pip -q install pyspark

### Comprobación de software
Si el inicio falla, registra el mensaje completo, la versión de Python y de Java. En ejecución local, Apache Spark requiere una instalación Java compatible con la versión elegida.

In [ ]:
import sys
print("Python:",sys.version.split()[0])
import pyspark
print("PySpark:",pyspark.__version__)

## 1. Reto de inicio: ¿qué preguntas debemos responder?
Una organización ficticia registra cumplimiento de metas diarias en plantas de agua, energía y residuos. Se sospecha que existen errores de captura de datos. Antes de generar un indicador gerencial, propón **dos preguntas verificables** y **dos reglas de calidad**.

**Respuesta del equipo:**
- Pregunta 1: ...
- Pregunta 2: ...
- Regla 1: ...
- Regla 2: ...

## 2. Crear una sesión Spark (ejemplo guiado)
`local[*]` utiliza los recursos disponibles de la máquina en un modo local; **no significa que se haya desplegado un clúster de varias máquinas**.

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("local[*]")
         .appName("UPN_BigData_S11_Intro")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)
print("Modo:", spark.sparkContext.master)

## 3. Cargar el archivo CSV
**Colab:** usa el panel lateral Archivos para subir el CSV; normalmente quedará en `/content/`. **Jupyter local:** coloca el CSV en la misma carpeta del notebook. El siguiente bloque detecta ambas opciones.

In [ ]:
from pathlib import Path
candidatos = [Path("registros_operativos_simulados.csv"),
              Path("/content/registros_operativos_simulados.csv")]
ruta = next((str(x) for x in candidatos if x.exists()), None)
assert ruta, "No se encontró el CSV. Súbelo en Colab o colócalo junto al notebook."
print("Archivo localizado:", ruta)
df = (spark.read.option("header", True)
      .option("inferSchema", True)
      .option("encoding", "UTF-8").csv(ruta))
df.printSchema()
df.show(8, truncate=False)
print("Filas cargadas:", df.count())

**Analiza:** ¿Qué tipos de dato asignó Spark a `meta_diaria` y `valor_observado`? ¿Qué riesgos implica confiar siempre en la inferencia de esquemas?

## 4. Demostración: seleccionar y filtrar (ejemplo guiado)
Prueba con una transformación y una acción, e identifica qué se ejecuta en cada línea.

In [ ]:
df_norte = df.filter(df.zona == "Norte").select("id_registro","sector","meta_diaria")
print("Se definió una transformación")
df_norte.show(5, truncate=False)

## 5. RETO A: diagnóstico de calidad
Completa las celdas para identificar (a) valores nulos, (b) metas negativas y (c) identificadores repetidos. No elimines registros automáticamente: explica primero el criterio de negocio.

**Evidencia:** tabla de hallazgos con cantidad, IDs afectados y recomendación.

In [ ]:
# RETO A.1: cuenta registros con valor observado nulo
# Pista: from pyspark.sql.functions import col
# Escribe tu código debajo:


In [ ]:
# RETO A.2: encuentra filas con meta_diaria <= 0
# Escribe tu código debajo:


In [ ]:
# RETO A.3: identifica IDs duplicados
# Pista: groupBy(...).count().filter(...)
# Escribe tu código debajo:


**Pregunta de decisión:** `R027` tiene un valor observado mayor que su meta. ¿Es necesariamente un error? Sustenta una regla de validación que distinga inconsistencia técnica de fenómeno legítimo.

## 6. RETO B: exploración y comparación
Propón una consulta en la API de DataFrames que permita comparar registros por sector. Primero define qué métrica utilizarás y qué registros excluirás (si corresponde). Luego presenta un resultado reproducible.

**Evita** interpretar las estadísticas como hechos reales: todos los datos son simulados.

In [ ]:
# RETO B: agrupación por sector
# Pista: df.groupBy("sector").count()
# Amplía el análisis con un indicador que tenga sentido para el problema.


## 7. RETO C: detectar y corregir un error lógico
El siguiente fragmento produce un indicador conceptualmente discutible: **promedia los porcentajes diarios** sin analizar reglas de calidad ni ponderación. Explica cuándo podría ser engañoso. Propón una alternativa y verifica su resultado.

```python
from pyspark.sql.functions import avg, col
indicador = df.withColumn("cumplimiento", col("valor_observado") / col("meta_diaria"))
indicador.groupBy("sector").agg(avg("cumplimiento")).show()
```

**No ejecutes sin analizar primero** el denominador no válido y los registros duplicados.

In [ ]:
# RETO C: escribe tu corrección justificada aquí


## 8. Pruebas de consistencia y reflexión
Antes de concluir, ejecuta la comprobación básica. Esta valida únicamente el archivo de entrenamiento original, **no valida tus decisiones de limpieza**.

In [ ]:
assert df.count() == 28, "El archivo fue alterado o no corresponde al suministrado"
assert len(df.columns) == 7
print("Estructura de archivo de práctica verificada")

**Discusión de cierre (responder con evidencia):**
1. ¿Qué ventaja representa Spark frente a una lectura convencional de CSV en Python? ¿En qué tamaño de datos sería injustificado afirmar que Spark es más rápido?
2. ¿Qué diferencia existe entre una transformación y una acción? Señala una línea de este notebook como evidencia.
3. ¿Por qué un conteo correcto no garantiza calidad de datos?
4. ¿Qué cambió en tu solución después de inspeccionar los registros anómalos?

**Entregables:** notebook ejecutado, cuadro de calidad de datos, criterios de corrección y conclusiones breves.

**Uso de IA:** si se utilizó, declarar herramienta, finalidad, código contrastado y correcciones realizadas; cada integrante debe poder explicar su propuesta.

**Reto opcional:** exporta a CSV los registros observados como anomalías y documenta tu criterio de selección. No sobrescribas el archivo original.